# Recipe-MPR QA — Demo Notebook

This notebook is the evaluator-facing walkthrough of the project. It shows the full input/output path for the multiple-choice recipe recommendation task end to end, using only the canonical APIs in `recipe_mpr_qa`.

**What you will see:**
1. Loading the prepared dataset and the deterministic train/val/test split.
2. Picking one test example and inspecting its query, candidate recipes, and gold answer.
3. Building the model-facing multiple-choice prompt.
4. Parsing a sample model response back to a recipe id and checking correctness.
5. Loading one of the saved evaluation result files and printing its accuracy summary.
6. (Optional) Running a small SmolLM2 model locally on the same example, if `torch` + `transformers` are installed.

**Setup:**
```bash
pip install -r requirements.txt
pip install -e .
recipe-mpr-qa prepare-data \
  --input data/500QA.json \
  --output data/processed/recipe_mpr_qa.jsonl \
  --split-output data/processed/primary_split.json
```

Then open this notebook from the repo root and run the cells in order.

## 1. Imports and paths

All paths are anchored at the repository root, so this notebook works as long as it lives at the top of the repo.

In [ ]:
from pathlib import Path

from recipe_mpr_qa.data.loaders import (
    load_dataset,
    load_split_manifest,
    get_split_examples,
)
from recipe_mpr_qa.formats import (
    DEFAULT_PROMPT_SPEC,
    build_multiple_choice_prompt,
    parse_multiple_choice_response,
)
from recipe_mpr_qa.evaluation.results import load_evaluation_result

REPO_ROOT = Path.cwd()
DATA_PATH = REPO_ROOT / "data" / "processed" / "recipe_mpr_qa.jsonl"
SPLIT_PATH = REPO_ROOT / "data" / "processed" / "primary_split.json"
RESULTS_DIR = REPO_ROOT / "llm_evaluation" / "results"

print(f"Repo root:   {REPO_ROOT}")
print(f"Dataset:     {DATA_PATH.exists()=}")
print(f"Split file:  {SPLIT_PATH.exists()=}")
print(f"Results dir: {RESULTS_DIR.exists()=}")

## 2. Load the prepared dataset and the test split

`load_dataset` reads the normalized JSONL with stable example ids (`rmpr-0001` ... `rmpr-0500`). `load_split_manifest` reads the deterministic 350/75/75 train/validation/test split (seed 1508, stratified by query type).

In [ ]:
dataset = load_dataset(DATA_PATH)
split_manifest = load_split_manifest(SPLIT_PATH)
test_examples = get_split_examples(dataset, split_manifest, "test")

print(f"Total examples in dataset: {len(dataset.examples)}")
print(f"Train / val / test sizes:  {len(split_manifest.splits['train'])} / {len(split_manifest.splits['validation'])} / {len(split_manifest.splits['test'])}")
print(f"Test split returned:       {len(test_examples)} examples")

## 3. Pick one test example

Each example contains a natural-language food preference query, exactly five candidate recipe descriptions, the id of the gold answer, and a set of query-type flags (Specific, Commonsense, Negated, Analogical, Temporal).

In [ ]:
example = test_examples[0]

print(f"example_id: {example.example_id}")
print(f"query:      {example.query}")
print()
print("options:")
for option in example.options:
    marker = "*" if option.option_id == example.answer_option_id else " "
    print(f"  {marker} [{option.option_id}] {option.text}")
print()
print(f"gold answer_option_id: {example.answer_option_id}")
active_flags = [name for name, on in example.query_type_flags.items() if on]
print(f"query types:           {active_flags or ['None']}")

## 4. Build the model-facing multiple-choice prompt

`build_multiple_choice_prompt` renders the canonical prompt (template version `recipe-mpr-mc-v1`) and returns both the prompt string and the letter→option_id mapping the model's response will be parsed against. We also pass `shuffle_key=example.example_id` so option order is shuffled deterministically per example, mitigating any positional bias the underlying model might have.

In [ ]:
prompt_text, letter_to_option_id = build_multiple_choice_prompt(
    query=example.query,
    options=example.options,
    prompt_spec=DEFAULT_PROMPT_SPEC,
    shuffle_key=example.example_id,
)

print(prompt_text)
print("\n--- letter -> option_id ---")
for letter, option_id in letter_to_option_id.items():
    is_gold = option_id == example.answer_option_id
    print(f"  {letter}: {option_id}{'  <-- gold' if is_gold else ''}")

## 5. Parse a sample model response and check correctness

`parse_multiple_choice_response` is the same parser used in production evaluation. It accepts bare letters, `"Option B"`-style replies, `"The answer is C."` patterns, and a few other common shapes seen in real model outputs. Here we synthesize a response so the demo runs without any actual model.

We construct a response that picks the gold letter, so the demo always demonstrates a correct end-to-end path. To see a wrong-answer path, change `simulated_gold_letter` to a different letter.

In [ ]:
gold_letter = next(
    letter
    for letter, option_id in letter_to_option_id.items()
    if option_id == example.answer_option_id
)

simulated_response = f"The best option is {gold_letter}."
parsed_letter = parse_multiple_choice_response(simulated_response)
predicted_option_id = letter_to_option_id.get(parsed_letter) if parsed_letter else None
is_correct = predicted_option_id == example.answer_option_id

print(f"simulated response:    {simulated_response!r}")
print(f"parsed letter:         {parsed_letter}")
print(f"predicted option_id:   {predicted_option_id}")
print(f"gold option_id:        {example.answer_option_id}")
print(f"correct:               {is_correct}")

## 6. Load a saved evaluation result and print its accuracy summary

Every model we evaluated has a JSON result file under `llm_evaluation/results/`. The headline result is `SmolLM2-1.7B-Instruct_finetuned_synthetic_test_75.json` (Run K) — SmolLM2-1.7B-Instruct fine-tuned with LoRA on the train split plus a 25% synthetic-data ratio, which scores **82.7%** on the held-out test set.

In [ ]:
result_path = RESULTS_DIR / "SmolLM2-1.7B-Instruct_finetuned_synthetic_test_75.json"
summary = load_evaluation_result(result_path)
print(summary.format_report())

### All evaluation results at a glance

Loop over the result files and print one line per model so you can see the full landscape — zero-shot SmolLM2 family, fine-tuned SmolLM2 family with and without synthetic data, fine-tuned Qwen2.5-3B, and the larger reference baselines (zero-shot Qwen2.5-3B and DeepSeek-R1-Distill-Qwen-7B).

In [ ]:
result_files = sorted(RESULTS_DIR.glob("*_test_75.json"))
print(f"{'file':<60s}  {'overall':>8s}  {'correct/total':>13s}")
print("-" * 86)
for result_file in result_files:
    summary = load_evaluation_result(result_file)
    print(
        f"{result_file.name:<60s}  {summary.overall_accuracy:>7.1%}  "
        f"{summary.total_correct:>5d}/{summary.total:<5d}"
    )

### Bar chart

The same numbers, plotted by model family with color coding for zero-shot vs fine-tuned vs fine-tuned-with-synthetic. This figure is the one that appears in the appendix of the final report.

In [ ]:
from IPython.display import Image, display

chart_path = RESULTS_DIR / "results_bar_chart.png"
if chart_path.exists():
    display(Image(filename=str(chart_path)))
else:
    print(f"Chart not found at {chart_path}. Regenerate it with:")
    print("  python llm_evaluation/plot_results.py")

## 7. (Optional) Run a small SmolLM2 model on the same example

This cell is **optional**. It downloads and runs `HuggingFaceTB/SmolLM2-135M-Instruct` (≈270 MB) and shows that the same prompt + parser path works on a real model. It is wrapped in a try/except so the rest of the notebook still runs cleanly without `torch` / `transformers` installed.

Note: SmolLM2-135M is too small to actually solve Recipe-MPR (it scores at random chance). The goal here is just to demonstrate the inference path, not to reproduce the headline accuracy.

In [ ]:
try:
    import torch  # noqa: F401
    from transformers import AutoModelForCausalLM, AutoTokenizer

    checkpoint = "HuggingFaceTB/SmolLM2-135M-Instruct"
    device = "cuda" if torch.cuda.is_available() else "cpu"

    tokenizer = AutoTokenizer.from_pretrained(checkpoint)
    model = AutoModelForCausalLM.from_pretrained(checkpoint).to(device)

    chat = [{"role": "user", "content": prompt_text}]
    chat_text = tokenizer.apply_chat_template(
        chat, tokenize=False, add_generation_prompt=True
    )
    enc = tokenizer(chat_text, return_tensors="pt").to(device)
    out = model.generate(
        **enc,
        max_new_tokens=8,
        do_sample=False,
        temperature=1.0,
        pad_token_id=tokenizer.eos_token_id,
    )
    new_tokens = out[0][enc["input_ids"].shape[1]:]
    response_text = tokenizer.decode(new_tokens, skip_special_tokens=True)

    parsed = parse_multiple_choice_response(response_text)
    predicted_id = letter_to_option_id.get(parsed) if parsed else None

    print(f"raw response:        {response_text!r}")
    print(f"parsed letter:       {parsed}")
    print(f"predicted option_id: {predicted_id}")
    print(f"gold option_id:      {example.answer_option_id}")
    print(f"correct:             {predicted_id == example.answer_option_id}")
except ImportError as exc:
    print("Skipping live inference: torch / transformers not installed.")
    print(f"  Reason: {exc}")
    print("  Install with: pip install torch transformers")
except Exception as exc:  # noqa: BLE001
    print(f"Skipping live inference: {type(exc).__name__}: {exc}")

## Where to go next

- **Reproduce the headline numbers:** see `README.md` → *Final Results* and `docs/final-report/report.tex`.
- **Re-run an evaluation from scratch:** `python llm_evaluation/mc_eval.py --model <hf-id-or-ollama-name> --backend {huggingface,ollama}`.
- **Re-run a fine-tune:** `python finetuning/finetune.py` (see `slurm/` for the cluster scripts used in the project).
- **Browse the docs hub:** `docs/index.md`.